# 2b - AI wrote it. Is it right?

**Time:** 35 minutes  |  **AI: OFF** (no Gemini, no ChatGPT in this notebook)

**What you will do**
- Read two functions written by an AI assistant: `portfolio_return` and `portfolio_vol`.
- Test them with three checks you can run without any code knowledge: a range check, a known case, and a valid input.
- Find and fix the three bugs. Then run a crisis.

This is the real job: AI writes code in seconds; you decide if the number is right. Each step has a **Stuck?** hint. Read it only after you have tried.

## The problem

Same funds A, B and C as in 2a. The client holds **20% in A, 30% in B and 50% in C**. A colleague asked an AI assistant for the return and the volatility of the portfolio. The code runs, with no error message.

| Step | Check | What the right code gives |
|---|---|---|
| 1 | Range check: return | between 5% and 9%: **7.6%** |
| 2 | Range check and known case: volatility | **18.44%**; all correlations 1 gives **21.5%** |
| 3 | A valid input | 70 / 20 / 10 gives **13.26%**, no error |
| 4 | Crisis: all correlations 0.9 | **20.88%** |

Run the cell with the assumptions first.

In [ ]:
import numpy as np

# The assumptions. Rates and weights are decimals.
w = np.array([0.20, 0.30, 0.50])      # client weights in A, B, C
ret = np.array([0.05, 0.07, 0.09])    # expected returns
vol = np.array([0.15, 0.20, 0.25])    # volatilities
corr = np.array([[1.0, 0.1, 0.4],
                 [0.1, 1.0, 0.7],
                 [0.4, 0.7, 1.0]])

## The AI's code

Read it, then run it.

**Your prediction:** is this code right? Look only at the two numbers it prints. Do they make sense for funds that return 5% to 9% with vols of 15% to 25%?

In [ ]:
# Code written by an AI assistant, for the spec: "return and volatility of a portfolio"

def portfolio_return(weights, returns):
    return np.mean(weights * returns)


def portfolio_vol(weights, vol, corr):
    if np.sum(weights) != 1:
        raise ValueError("Weights must sum to 1.")
    cov = np.diag(vol) @ corr
    return np.sqrt(weights @ cov @ weights)


print(f"Portfolio return: {portfolio_return(w, ret):.1%}")
print(f"Portfolio vol:    {portfolio_vol(w, vol, corr):.1%}")

## Step 1 - Range check on the return

A **range check**: a weighted average must sit between the smallest and the largest value. The funds return 5%, 7% and 9%, so the portfolio must return between 5% and 9%. The AI's code gives 2.5%.

Find the wrong line and fix it in the cell below. You should get **7.6%**.

**Stuck?** The return of the portfolio is weight x return for each fund, **added up**. What does `np.mean` do that `np.sum` does not?

In [ ]:
def portfolio_return(weights, returns):
    return np.mean(weights * returns)    # fix this line


print(f"Portfolio return: {portfolio_return(w, ret):.1%}")

## Step 2 - Range check and known case on the volatility

The AI's volatility is 39.0%, above the riskiest fund (25%). Mixing funds can never be riskier than the riskiest one. Something is wrong, but where?

Use a **known case**, a situation where you know the answer without code: if all correlations are 1, the portfolio vol must be the weighted average of the vols, **21.5%** exactly. Run the cell below once as it is, and look at the second line. Then find the wrong line and fix it.

You should get **18.44%** for the client and **21.50%** for the known case. (Checks card, entry 5.)

**Stuck?** The covariance of two funds is the vol of the first, times their correlation, times the vol of the second. Count the vols in the line `cov = ...`.

In [ ]:
def portfolio_vol(weights, vol, corr):
    if np.sum(weights) != 1:
        raise ValueError("Weights must sum to 1.")
    cov = np.diag(vol) @ corr            # fix this line
    return np.sqrt(weights @ cov @ weights)


print(f"Client portfolio:   {portfolio_vol(w, vol, corr):.2%}")
print(f"All correlations 1: {portfolio_vol(w, vol, np.ones((3, 3))):.2%}")

## Step 3 - A valid input

The fixed function works for the client. Now try another real portfolio: **70% A, 20% B, 10% C**. Those weights sum to 100%, so the function should accept them.

Run the cell below. **Your prediction:** what happens?

In [ ]:
print(0.7 + 0.2 + 0.1)
print(0.7 + 0.2 + 0.1 == 1)
print(round(0.7 + 0.2 + 0.1, 6) == 1)

`0.7 + 0.2 + 0.1` gives `0.9999999999999999`: computers store most decimals with tiny errors. So the AI's guard `np.sum(weights) != 1` stops a **valid** portfolio with "Weights must sum to 1." **Never test decimals with `==` or `!=`.** Round first.

Fix the guard in the cell below. You should get **13.26%** for 70 / 20 / 10, and still **18.44%** for the client.

**Stuck?** Look at the third line printed above. Use the same idea in the `if`: round the sum to 6 decimals before you compare it with 1.

In [ ]:
def portfolio_vol(weights, vol, corr):
    if np.sum(weights) != 1:             # fix this line
        raise ValueError("Weights must sum to 1.")
    cov = np.diag(vol) @ corr @ np.diag(vol)
    return np.sqrt(weights @ cov @ weights)


print(f"70 / 20 / 10:     {portfolio_vol(np.array([0.7, 0.2, 0.1]), vol, corr):.2%}")
print(f"Client portfolio: {portfolio_vol(w, vol, corr):.2%}")

## Behind? Catch up here

If steps 1 to 3 do not work yet, run the cell below: it holds the fixed functions, so you can do the crisis. Then go back to your fixes later.

In [ ]:
# CATCH-UP CELL: the fixed functions
def portfolio_return(weights, returns):
    return np.sum(weights * returns)


def portfolio_vol(weights, vol, corr):
    if round(np.sum(weights), 6) != 1:
        raise ValueError("Weights must sum to 1.")
    cov = np.diag(vol) @ corr @ np.diag(vol)
    return np.sqrt(weights @ cov @ weights)


print("Ready for step 4.")

## Step 4 - Crisis: all correlations 0.9

In a crisis, funds fall together. Build a correlation matrix with 0.9 everywhere, then put 1 on the diagonal (a fund is always perfectly correlated with itself).

- `np.full((3, 3), 0.9)` makes a 3 x 3 array filled with 0.9.
- `np.fill_diagonal(crisis, 1)` sets the diagonal to 1. It changes `crisis` itself, so do not assign its result to anything.

You should get **20.88%**, almost back to the weighted average of 21.5%. Diversification disappears when you need it most.

In [ ]:
crisis = ...   # YOUR CODE
...            # YOUR CODE

print(crisis)
print(f"Crisis vol: {portfolio_vol(w, vol, crisis):.2%}")

## Go further (if you finish early)

### Go further 1 - How much does diversification save?

Write a function `diversification_saving(weights, vol, corr)` that returns the weighted average of the vols minus the portfolio vol. Client: **3.06 points**. Crisis: **0.62 points**.

In [ ]:
# YOUR CODE

### Go further 2 - At what correlation is half the saving gone?

Set every correlation to the same value `rho` (1 on the diagonal), for `rho` from 0 to 1 in steps of 0.05, with a for loop. At which `rho` does the client portfolio save less than **half** of what it saves at `rho = 0`? (Answer: **0.45**.)

In [ ]:
# YOUR CODE

### Go further 3 - Plant a bug for your neighbour

Copy the fixed `portfolio_vol` into the cell below and hide **one** bug in it that still runs without an error. Give it to your neighbour: which check catches it?

In [ ]:
# Your function with one hidden bug

## What you learned

**Python**
- **Function** on arrays: `portfolio_return` and `portfolio_vol` take **arrays** and give back a number. One formula, any portfolio.
- **Sum** vs **mean**: `np.sum(w * ret)` is the weighted return; `np.mean` divides by the number of funds.
- **Matrix product** (`weights @ cov @ weights`), **Diagonal matrix** (`np.diag(vol)`, on **both** sides of `corr`) and **Square root** (`np.sqrt`).
- **Guard**: `if round(np.sum(weights), 6) != 1: raise ValueError(...)`. **round** first: never test decimals with `==` or `!=`.
- **2D array** built with `np.ones((3, 3))`, `np.full((3, 3), 0.9)` and `np.fill_diagonal`.

**Finance**
- Code that runs is not code that is right. The AI's code gave 2.5% and 39.0%, with no warning.
- Three checks found three bugs: a **range check** (a weighted average sits between the smallest and the largest value), a **known case** (all correlations 1 gives the weighted average, 21.5%), and a **valid input** (70 / 20 / 10 must be accepted).
- The client portfolio: return 7.6%, vol 18.44% against 21.5% for the weighted average. Crisis: 20.88%. Diversification disappears when you need it most.

Keep two pages open from now on:
- [`python-card.md`](https://github.com/niparis/gem-python-finance/blob/main/python-card.md) - what each piece of Python means
- [`checks-card.md`](https://github.com/niparis/gem-python-finance/blob/main/checks-card.md) - how to know a number is right